# Zero-Shot Video Anomaly Detection: Shard Precomputation on Colab T4

This notebook runs embedding precomputation for **shard 1/4** of UCF-Crime test set videos.
- Dual model pass in single decode: `google/siglip2-base-patch16-384` and `openai/clip-vit-base-patch32` (A0 baseline).
- Standardized sampling rate: `--fps 1`.
- Self-contained: mounts no external drive by default.
- Repository: `https://github.com/Kyosaki6/vlm-anomalies-detection.git`

In [ ]:
# 1. Hardware verification: Check CUDA GPU (Tesla T4)
!nvidia-smi


In [ ]:
# 2. Clone repository and change working directory
!git clone https://github.com/Kyosaki6/vlm-anomalies-detection.git
%cd vlm-anomalies-detection


In [ ]:
# 3. Install dependencies
!pip install -q -r requirements-dev.txt


In [ ]:
# 4. Check manifest availability (create fallback stub for dry-run if not yet generated)
from pathlib import Path
manifest_file = Path("manifests/ucf_test.csv")
if not manifest_file.exists():
    print("Notice: manifests/ucf_test.csv not found. Generating sample manifest for dry-run verification...")
    manifest_file.parent.mkdir(parents=True, exist_ok=True)
    with open(manifest_file, "w", encoding="utf-8") as f:
        f.write("video_id,path,dataset,split,n_frames,fps\n")
        f.write("Abuse028_x264,datasets/ucf_crime/test/Abuse028_x264.mp4,ucf_crime,test,1400,30.0\n")
        f.write("Arrest001_x264,datasets/ucf_crime/test/Arrest001_x264.mp4,ucf_crime,test,1200,30.0\n")
        f.write("Normal_Videos001_x264,datasets/ucf_crime/test/Normal_Videos001_x264.mp4,ucf_crime,test,1500,30.0\n")
        f.write("Normal_Videos002_x264,datasets/ucf_crime/test/Normal_Videos002_x264.mp4,ucf_crime,test,1800,30.0\n")
    print("Sample manifest written to manifests/ucf_test.csv")
else:
    print(f"Found existing manifest: {manifest_file}")


## Shard Precomputation: Dry-Run Cell (Default)

Verifies shard assignment and planning without loading GPU models or decoding video files.

In [ ]:
# 5. DRY-RUN: Precompute planning for Shard 1/4
!python scripts/precompute_embeddings.py \
    --manifest manifests/ucf_test.csv \
    --shard 1/4 \
    --fps 1 \
    --model google/siglip2-base-patch16-384 \
    --model openai/clip-vit-base-patch32 \
    --dry-run


## Shard Precomputation: Real Run (Execute with Datasets)

Run this cell when video files are present in `datasets/ucf_crime/test/` to compute real embeddings.
Uses `float16` for Colab Tesla T4 GPU.

In [ ]:
# 6. REAL RUN: Compute and save embeddings for Shard 1/4
!python scripts/precompute_embeddings.py \
    --manifest manifests/ucf_test.csv \
    --shard 1/4 \
    --fps 1 \
    --dtype float16 \
    --model google/siglip2-base-patch16-384 \
    --model openai/clip-vit-base-patch32


## Package and Export Cached Embeddings

Zip `cache/embeddings/` to download and merge with other machine shards.

In [ ]:
# 7. Zip cache/embeddings for transfer
!mkdir -p cache/embeddings
!zip -r cache_embeddings_shard1.zip cache/embeddings/
print("Cached embeddings packaged to cache_embeddings_shard1.zip")
